# Principal Component Analysis: From Statistics to Geometry

In this notebook, we bridge the gap between statistics and linear algebra. We will rigorously derive Principal Component Analysis (PCA) not as a black-box algorithm, but as a natural consequence of asking a geometric question: *How can we describe a data cloud in a way that preserves the maximum amount of variation?*

We build directly on the linear algebra concepts established previously:
- Vectors and matrices as geometric transformations
- Projections onto vectors and subspaces
- Positive semidefinite matrices and quadratic forms
- The interpretation of symmetric diagonalization: $A = Q\Lambda Q^\intercal$

We already understand that $Q\Lambda Q^\intercal$ represents:
1. $Q^\intercal x$: Express $x$ in the orthonormal eigenvector basis.
2. $\Lambda(Q^\intercal x)$: Independently scale each coordinate by its eigenvalue.
3. $Q\Lambda Q^\intercal x$: Transform the result back into the original coordinate system.

Our goal is not to reteach linear algebra from scratch, but to derive exactly why PCA works, starting from data and variance, all the way to orthogonal diagonalization and dimensionality reduction.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic style
sns.set_theme(style="whitegrid")
np.random.seed(42)

# Setup path for local imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


## Part 1: Data as Geometry

Start by establishing the data convention.
We define our dataset $X \in \mathbb{R}^{n \times d}$ where:
- $n$ = number of samples
- $d$ = number of features
- each row represents one sample
- each column represents one feature

$$
X = \begin{bmatrix}
x_1^\intercal \\
x_2^\intercal \\
\vdots \\
x_n^\intercal
\end{bmatrix}
$$

A dataset is not merely a table of numbers. Geometrically, each sample is a point in $\mathbb{R}^d$.
- Two features $\rightarrow$ points in a 2D plane
- Three features $\rightarrow$ points in 3D space
- 784 image pixels $\rightarrow$ a single image is a point in $\mathbb{R}^{784}$

Let's visualize a small 2D dataset as a cloud of points.


In [ ]:
# Generate a simple 2D point cloud
x_points = np.random.normal(loc=5, scale=2, size=100)
y_points = 0.5 * x_points + np.random.normal(loc=2, scale=1, size=100)
X_demo = np.column_stack((x_points,y_points))

plt.figure(figsize=(6, 6))
plt.scatter(x_points, y_points, alpha=0.6, edgecolors='k')
plt.title("A Dataset as a Geometric Cloud of Points in $\mathbb{R}^2$")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.axis('equal')
plt.show()


This visualization establishes our central question:
**"Can we describe this cloud using fewer dimensions while preserving as much important structure as possible?"**


## Part 2: The Mean as the Center of the Data Cloud

We begin by identifying the center of this cloud. The sample mean vector $\mu \in \mathbb{R}^d$ is defined as the average of all samples:

$$ \mu = \frac{1}{n} \sum_{i=1}^n x_i $$

Geometrically, $\mu$ is the center of mass of the data cloud. The vector from the mean to any data point $x_i$ represents how that specific point deviates from the center. These deviations are crucial because PCA is fundamentally about studying the *spread* or *variation* around this center.


In [ ]:
mu_demo = np.mean(X_demo, axis=0)

plt.figure(figsize=(6, 6))
plt.scatter(X_demo[:, 0], X_demo[:, 1], alpha=0.5, label="Data points")
plt.scatter(mu_demo[0], mu_demo[1], color='red', s=150, marker='X', label="Mean $\mu$")

# Draw a few deviation vectors
for i in range(5):
    plt.arrow(mu_demo[0], mu_demo[1], X_demo[i, 0] - mu_demo[0], X_demo[i, 1] - mu_demo[1], 
              color='k', alpha=0.3, width=0.02, head_width=0.15, length_includes_head=True)

plt.title("The Mean and Deviations from the Center")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.axis('equal')
plt.show()


## Part 3: Mean Centering

We define centered samples by shifting the origin of our coordinate system to the mean $\mu$:

$$ x^c_i = x_i - \mu $$

To write this in matrix notation for the entire dataset, let $\mathbf{1} \in \mathbb{R}^{n \times 1}$ be a column vector of ones. Since our samples are rows in $X$, the mean vector $\mu$ is a row vector in $\mathbb{R}^{1 \times d}$. 

$$ X_c = X - \mathbf{1}\mu^\intercal $$

Let's check the dimensions: $\mathbf{1}$ is $(n \times 1)$ and $\mu^\intercal$ is $(1 \times d)$. Their outer product $\mathbf{1}\mu^\intercal$ is an $(n \times d)$ matrix where every row is exactly $\mu$. Thus, we are subtracting the mean from every single sample simultaneously.

**Is centering only used to simplify formulas?**
No! Centering is not merely an algebraic convenience. PCA studies the variation *around the center* of the data. 
Covariance itself is defined using deviations from the mean. If we compute the second-moment matrix without centering ($X^\intercal X$), the principal directions will be heavily influenced by the distance of the cloud from the origin, rather than the intrinsic shape of the cloud itself. By shifting the data to the origin, the geometry of the covariance matrix perfectly describes the spread of the data.

Furthermore, the centered data has a mean of exactly zero:
$$ \sum_{i=1}^n x^c_i = \sum_{i=1}^n (x_i - \mu) = \left( \sum_{i=1}^n x_i \right) - n\mu = n\mu - n\mu = 0 $$


In [ ]:
X_centered = X_demo - mu_demo

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].scatter(X_demo[:, 0], X_demo[:, 1], alpha=0.6, color='blue')
axes[0].scatter(mu_demo[0], mu_demo[1], color='red', s=100, marker='X')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].axvline(0, color='black', lw=0.5)
axes[0].set_title("Original Data Cloud (away from origin)")
axes[0].axis('equal')

axes[1].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.6, color='green')
axes[1].scatter(0, 0, color='red', s=100, marker='X')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].axvline(0, color='black', lw=0.5)
axes[1].set_title("Centered Data Cloud (mean at origin)")
axes[1].axis('equal')

plt.show()


## Part 4 & 5: Variance and Degrees of Freedom

Let's begin with a one-dimensional variable $x$. The sample variance is defined as:

$$ \text{Var}(x) = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2 $$

Using our centered vector $x_c \in \mathbb{R}^n$, we can rewrite this sum of squares as a dot product:

$$ x_c = \begin{bmatrix} x_1 - \bar{x} \\ \vdots \\ x_n - \bar{x} \end{bmatrix} $$

$$ \text{Var}(x) = \frac{1}{n-1} x_c^\intercal x_c = \frac{1}{n-1} ||x_c||^2 $$

**Geometric Interpretation:**
Variance is the squared Euclidean length (or "energy") of the centered data vector, normalized by the degrees of freedom. 
Note the distinction: $||x_c||$ is the norm, $||x_c||^2$ is the squared norm (sum of squared deviations), and variance is the squared norm divided by $n-1$.

**Why divide by $n-1$?**
We use $n-1$ instead of $n$ because of a constraint on the residuals. Since we know that:
$$ \sum_{i=1}^n (x_i - \bar{x}) = 0 $$
if we are given $n-1$ centered deviations, the final $n$-th deviation is completely constrained (it must be whatever value makes the sum zero). Therefore, the centered deviations cannot vary independently in all $n$ directions. We have lost one *degree of freedom* because we used the data itself to estimate the mean $\bar{x}$.
Dividing by $n$ would systematically underestimate the population variance (Bessel's correction).


## Part 6: Covariance

Now consider two variables, represented by centered vectors $x_c$ and $y_c$.
The sample covariance is defined as:

$$ \text{Cov}(x,y) = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y}) = \frac{1}{n-1} x_c^\intercal y_c $$

Notice the beautiful symmetry:
- **Variance** is the *self dot-product*: $\text{Var}(x) = \frac{1}{n-1} x_c^\intercal x_c$
- **Covariance** is the *cross dot-product*: $\text{Cov}(x,y) = \frac{1}{n-1} x_c^\intercal y_c$

The dot product measures alignment. 
- If $x_c$ and $y_c$ tend to have the same sign at the same observations, their dot product is large and positive $\rightarrow$ **Strong positive covariance**.
- If one tends to be positive while the other is negative, the dot product is negative $\rightarrow$ **Strong negative covariance**.
- If positive and negative products cancel out $\rightarrow$ **Near-zero covariance**.

Since $x_c^\intercal y_c = y_c^\intercal x_c$, covariance is symmetric: $\text{Cov}(x,y) = \text{Cov}(y,x)$.

*Crucial subtlety*: Covariance measures how deviations co-vary linearly. Zero covariance means the dot product is zero (orthogonal deviations), but it does not strictly imply statistical independence. For example, points perfectly forming a circle have zero covariance, but $y$ is entirely dependent on $x$.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Positive covariance
x_pos = np.random.normal(0, 1, 100)
y_pos = x_pos + np.random.normal(0, 0.5, 100)
axes[0].scatter(x_pos, y_pos, color='blue', alpha=0.6)
axes[0].set_title(f"Positive Covariance\nCov ~ {np.cov(x_pos, y_pos)[0,1]:.2f}")

# Negative covariance
x_neg = np.random.normal(0, 1, 100)
y_neg = -x_neg + np.random.normal(0, 0.5, 100)
axes[1].scatter(x_neg, y_neg, color='red', alpha=0.6)
axes[1].set_title(f"Negative Covariance\nCov ~ {np.cov(x_neg, y_neg)[0,1]:.2f}")

# Zero covariance
x_zero = np.random.normal(0, 1, 100)
y_zero = np.random.normal(0, 1, 100)
axes[2].scatter(x_zero, y_zero, color='green', alpha=0.6)
axes[2].set_title(f"Near-Zero Covariance\nCov ~ {np.cov(x_zero, y_zero)[0,1]:.2f}")

plt.show()


## Part 7: The Covariance Matrix

For a multivariate dataset with $d$ features, we compute the covariance between every pair of features, forming a $d \times d$ matrix $\Sigma$:

$$ \Sigma_{ij} = \text{Cov}(\text{feature}_i, \text{feature}_j) $$

For a 2D dataset with features $x$ and $y$:
$$
\Sigma = \begin{bmatrix}
\text{Var}(x) & \text{Cov}(x,y) \\
\text{Cov}(y,x) & \text{Var}(y)
\end{bmatrix}
$$
- The **diagonal** contains the variance of each individual feature.
- The **off-diagonal** contains the covariance between different features.

We can compute this matrix compactly using matrix multiplication:
$$ \Sigma = \frac{1}{n-1} X_c^\intercal X_c $$

**Derivation:**
Recall that $X_c$ is an $n \times d$ matrix where each column represents a centered feature across all samples.
When we multiply $X_c^\intercal$ ($d \times n$) by $X_c$ ($n \times d$), the resulting $d \times d$ matrix has the dot product of the $i$-th feature column and the $j$-th feature column at entry $(i,j)$. Dividing by $n-1$ normalizes these dot products into variances and covariances!


## Part 8: Properties of the Covariance Matrix

$\Sigma$ has two incredibly important linear algebraic properties:

**1. Symmetric**
$$ \Sigma^\intercal = \left( \frac{1}{n-1} X_c^\intercal X_c \right)^\intercal = \frac{1}{n-1} X_c^\intercal (X_c^\intercal)^\intercal = \frac{1}{n-1} X_c^\intercal X_c = \Sigma $$
Statistically, this reflects the fact that $\text{Cov}(x,y) = \text{Cov}(y,x)$.

**2. Positive Semidefinite**
For any arbitrary non-zero vector $z \in \mathbb{R}^d$:
$$ z^\intercal \Sigma z = z^\intercal \left( \frac{1}{n-1} X_c^\intercal X_c \right) z = \frac{1}{n-1} (X_c z)^\intercal (X_c z) = \frac{1}{n-1} ||X_c z||^2 \ge 0 $$
Because a squared norm cannot be negative, $z^\intercal \Sigma z$ is always $\ge 0$.

Because $\Sigma$ is a real, symmetric matrix, the **Spectral Theorem** from Notebook 01 guarantees that we can orthogonally diagonalize it:
$$ \Sigma = Q \Lambda Q^\intercal $$
where $Q$ is an orthogonal matrix ($Q^\intercal Q = I$) containing eigenvectors, and $\Lambda$ is a diagonal matrix containing real eigenvalues $\lambda_i \ge 0$.


## Part 9: Covariance as the Shape of a Data Cloud

Let's look at 2D datasets. The covariance matrix encodes spread, orientation, and correlation structure perfectly.
- A circular cloud has a diagonal covariance matrix with equal variances.
- An elongated horizontal cloud has a diagonal matrix with a large variance for $X$ and small for $y$.
- An elongated diagonal cloud has off-diagonal covariance entries (correlation).

The eigenvectors of $\Sigma$ point along the principal axes of the ellipse formed by the data cloud. The eigenvalues correspond to the variance along those axes. 


In [ ]:
from matplotlib.patches import Ellipse
import numpy.linalg as LA

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_cov_ellipse(ax, X_c, title):
    cov = np.cov(X_c, rowvar=False)
    evals, evecs = LA.eigh(cov)
    
    ax.scatter(X_c[:, 0], X_c[:, 1], alpha=0.5, zorder=1)
    
    # Plot eigenvectors
    for i in range(2):
        v = evecs[:, i] * np.sqrt(evals[i]) * 2
        ax.arrow(0, 0, v[0], v[1], color='red', width=0.05, head_width=0.2, zorder=3)
        
    ax.set_title(title)
    ax.axis('equal')
    ax.grid(True)
    ax.set_xlim(-6, 6)
    ax.set_ylim(-6, 6)

# Circular
X_circ = np.random.normal(0, 1, (200, 2))
plot_cov_ellipse(axes[0], X_circ, "Circular Cloud (Equal Variance, Uncorrelated)")

# Horizontal Elongated
X_horiz = np.random.normal(0, 1, (200, 2)) * np.array([3, 0.5])
plot_cov_ellipse(axes[1], X_horiz, "Horizontal Cloud (Uncorrelated)")

# Diagonal Elongated
theta = np.pi / 4
rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X_diag = X_horiz @ rot.T
plot_cov_ellipse(axes[2], X_diag, "Diagonal Cloud (Correlated)")

plt.show()


We can *see* that the red eigenvectors point in the directions of maximum and minimum spread. But how can we formally prove that these eigenvector directions are exactly the directions of maximum variance? Let's build up to it.


## Part 10: Projection onto an Arbitrary Direction

Let $w \in \mathbb{R}^d$ be a unit vector, so $w^\intercal w = 1$.
If we project a single centered sample $x^c$ onto the direction $w$, its scalar coordinate along that line is given by the dot product:
$$ z = w^\intercal x^c $$

For the entire dataset (where samples are rows of $X_c$), the projected coordinates for all $n$ samples can be computed at once:
$$ z_{data} = X_c w $$
Here, $X_c$ is $(n \times d)$ and $w$ is $(d \times 1)$, so the result is an $(n \times 1)$ column vector of scalar coordinates.

**What is the variance of this projected 1D dataset?**
By the definition of variance (for centered data):
$$ \text{Var}(\text{projected}) = \frac{1}{n-1} z_{data}^\intercal z_{data} $$
Substitute $z_{data} = X_c w$:
$$ \text{Var}(\text{projected}) = \frac{1}{n-1} (X_c w)^\intercal (X_c w) $$
$$ = \frac{1}{n-1} (w^\intercal X_c^\intercal) (X_c w) $$
$$ = w^\intercal \left( \frac{1}{n-1} X_c^\intercal X_c \right) w $$
$$ = w^\intercal \Sigma w $$

This is an incredible result! The variance of the dataset projected along ANY direction $w$ is exactly given by the quadratic form $w^\intercal \Sigma w$. This beautifully links statistics (variance), geometry (projection), and linear algebra (quadratic forms).


## Part 11: The PCA Optimization Problem

We want to find the direction $w$ that captures the most information. Statistically, this means finding the direction $w$ that maximizes the projected variance.

**The Optimization Problem:**
$$ \text{Maximize: } f(w) = w^\intercal \Sigma w $$
$$ \text{Subject to: } w^\intercal w = 1 $$

**Why the unit constraint?**
If we don't constrain the length of $w$, the problem is ill-posed. Suppose we just scale $w$ by a constant $c > 1$. The new variance would be:
$$ (cw)^\intercal \Sigma (cw) = c^2 (w^\intercal \Sigma w) $$
We could make the variance infinitely large just by making the vector $w$ longer! By enforcing $w^\intercal w = 1$, we ensure we are only searching over *directions*.


## Part 12: Lagrange Multipliers & the First Principal Component

To solve constrained optimization problems, we use Lagrange multipliers. We combine our objective function and constraint into a single Lagrangian function:

$$ \mathcal{L}(w, \lambda) = w^\intercal \Sigma w - \lambda(w^\intercal w - 1) $$
Here, $\lambda$ is a scalar Lagrange multiplier.

To find the maximum, we take the gradient with respect to the vector $w$ and set it to zero.
Recall from matrix calculus that because $\Sigma$ is symmetric:
- $\nabla_w (w^\intercal \Sigma w) = 2\Sigma w$
- $\nabla_w (w^\intercal w) = 2w$

Therefore:
$$ \nabla_w \mathcal{L} = 2\Sigma w - 2\lambda w = 0 $$
$$ \Sigma w = \lambda w $$

**This is a massive breakthrough.**
The optimization problem has turned exactly into the defining equation of an **eigenvector**! The optimal direction $w$ must be an eigenvector of the covariance matrix $\Sigma$.

But which eigenvector? Let's find the variance along this optimal direction. Multiply both sides by $w^\intercal$:
$$ w^\intercal \Sigma w = w^\intercal (\lambda w) $$
$$ w^\intercal \Sigma w = \lambda (w^\intercal w) $$

Since we constrained $w$ to be a unit vector ($w^\intercal w = 1$):
$$ w^\intercal \Sigma w = \lambda $$

The projected variance along the eigenvector $w$ is exactly equal to its eigenvalue $\lambda$!
Therefore, to *maximize* the variance, we must choose the eigenvector corresponding to the **largest eigenvalue**. 

This direction is the **first principal component**.


## Part 13: The Second and Later Principal Components

We can't just pick the first principal component again. We want the *next* direction that explains the remaining variance, subject to providing *new* information. Statistically, new information means the new coordinates should be uncorrelated with the first. Geometrically, this means the new direction must be orthogonal to the first.

**Optimization for $w_2$:**
$$ \text{Maximize: } w_2^\intercal \Sigma w_2 $$
$$ \text{Subject to: } w_2^\intercal w_2 = 1 \text{ AND } w_2^\intercal w_1 = 0 $$

Using a similar Lagrange multiplier derivation, the solution is the eigenvector corresponding to the second largest eigenvalue.

**Why does orthogonality guarantee zero covariance?**
Suppose we project the data onto two distinct eigenvectors $q_i$ and $q_j$. The projected coordinates are $z_i = X_c q_i$ and $z_j = X_c q_j$.
The covariance between these two new variables is:
$$ \text{Cov}(z_i, z_j) = \frac{1}{n-1} z_i^\intercal z_j = \frac{1}{n-1} (X_c q_i)^\intercal (X_c q_j) = q_i^\intercal \left( \frac{1}{n-1} X_c^\intercal X_c \right) q_j = q_i^\intercal \Sigma q_j $$

Since $q_j$ is an eigenvector ($\Sigma q_j = \lambda_j q_j$):
$$ \text{Cov}(z_i, z_j) = q_i^\intercal (\lambda_j q_j) = \lambda_j (q_i^\intercal q_j) $$

Because symmetric matrices have orthogonal eigenvectors, $q_i^\intercal q_j = 0$ for $i \neq j$.
Thus, $\text{Cov}(z_i, z_j) = 0$.
The principal component coordinates are perfectly uncorrelated!


## Part 14: Eigendecomposition of the Covariance Matrix

Because $\Sigma$ is symmetric, we can decompose it into:
$$ \Sigma = Q \Lambda Q^\intercal $$

Where:
- $Q = [q_1, q_2, \dots, q_d]$ is an orthogonal matrix whose columns are orthonormal eigenvectors.
- $\Lambda = \text{diag}(\lambda_1, \lambda_2, \dots, \lambda_d)$ is a diagonal matrix of real eigenvalues.

By convention, we sort the eigenvalues in descending order:
$$ \lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d \ge 0 $$
and we reorder the columns of $Q$ to match.
- $q_1$ is the greatest variance direction (PC1), with variance $\lambda_1$.
- $q_2$ is the next greatest independent direction (PC2), with variance $\lambda_2$.


## Part 15: PCA as a Change of Basis

PCA is fundamentally a coordinate transformation a change of basis.
For a single centered sample $x_c$, its representation in the new principal component basis is:
$$ z = Q^\intercal x_c $$

For the entire row-wise dataset $X_c$:
$$ Z = X_c Q $$
(If $x_c$ is a row, $x_c Q$ projects it onto the columns of $Q$).

Let's compute the covariance matrix of this completely transformed dataset $Z$:
$$ \text{Cov}(Z) = \frac{1}{n-1} Z^\intercal Z = \frac{1}{n-1} (X_c Q)^\intercal (X_c Q) $$
$$ = Q^\intercal \left( \frac{1}{n-1} X_c^\intercal X_c \right) Q $$
$$ = Q^\intercal \Sigma Q $$

Substitute $\Sigma = Q \Lambda Q^\intercal$:
$$ \text{Cov}(Z) = Q^\intercal (Q \Lambda Q^\intercal) Q = (Q^\intercal Q) \Lambda (Q^\intercal Q) $$
Since $Q$ is orthogonal ($Q^\intercal Q = I$):
$$ \text{Cov}(Z) = I \Lambda I = \Lambda $$

This is a central achievement: **PCA rotates the coordinate system so that the new covariance matrix is perfectly diagonal.** All off-diagonal elements (covariances) are exactly zero.


## Part 16: Dimensionality Reduction

What if we don't want all $d$ dimensions? We can truncate $Q$.
Suppose we keep only the first $k$ eigenvectors (the ones with the largest eigenvalues). Let $Q_k \in \mathbb{R}^{d \times k}$.

For the dataset, our low-dimensional projection is:
$$ Z = X_c Q_k $$

We have reduced our data from $n \times d$ to $n \times k$. 
Geometrically, we are projecting the data onto the $k$-dimensional subspace spanned by $\{q_1, \dots, q_k\}$. 
Because of our optimization problem, this specific subspace preserves the maximum possible variance among all possible $k$-dimensional subspaces.


## Part 17: Reconstruction

Can we get our original data back from the $k$-dimensional representation $Z$?
To map the coordinates $z \in \mathbb{R}^k$ back into the original feature space $\mathbb{R}^d$:
$$ \hat{x}_c = Q_k z $$

Substitute $z = Q_k^\intercal x_c$:
$$ \hat{x}_c = Q_k Q_k^\intercal x_c $$

Then we add the mean back to un-center the data:
$$ \hat{x} = Q_k Q_k^\intercal (x - \mu) + \mu $$

**Is $\hat{x} = x$?**
If $k = d$ (we keep all components), then $Q$ is a square orthogonal matrix, meaning $Q Q^\intercal = I$. In this case, reconstruction is perfect.
However, if $k < d$, $Q_k$ is rectangular. $Q_k^\intercal Q_k = I_k$ (identity in $\mathbb{R}^k$), but $Q_k Q_k^\intercal \neq I_d$. 
Instead, $Q_k Q_k^\intercal$ is a *projection matrix* onto the $k$-dimensional subspace.
Reconstruction is approximate. We have permanently lost the information that existed in the discarded directions.


## Part 18: Reconstruction Error

The error for a single sample is the squared Euclidean distance between the original and reconstructed point: $||x - \hat{x}||^2$.
For the whole dataset, the Mean Squared Error (MSE) is:
$$ \text{MSE}_{recon} = \frac{1}{n} \sum_{i=1}^n ||x_i - \hat{x}_i||^2 $$

Because of the orthogonality of the basis, the variance we lose is exactly the variance along the directions we threw away.
The total reconstruction error over the dataset is proportional to the sum of the discarded eigenvalues:
$$ \sum_{j=k+1}^d \lambda_j $$


## Part 19: Trace and Total Variance

Recall that the trace of a matrix is the sum of its diagonal elements.
The trace of the covariance matrix $\Sigma$ is the sum of the variances of the original features. This is the **total variance** of the dataset.

Using the cyclic property of the trace ($\text{tr}(ABC) = \text{tr}(CAB)$):
$$ \text{tr}(\Sigma) = \text{tr}(Q \Lambda Q^\intercal) = \text{tr}(\Lambda Q^\intercal Q) = \text{tr}(\Lambda I) = \text{tr}(\Lambda) = \sum_{i=1}^d \lambda_i $$

The total variance of the data is precisely the sum of the eigenvalues! PCA doesn't change the total variance; it simply redistributes it into orthogonal directions.


## Part 20: Explained Variance

We define the **Explained Variance Ratio** (EVR) for the $i$-th principal component as the fraction of total variance it captures:

$$ \text{EVR}_i = \frac{\lambda_i}{\sum_{j=1}^d \lambda_j} $$

The **Cumulative Explained Variance** for the first $k$ components is:
$$ \frac{\sum_{i=1}^k \lambda_i}{\sum_{j=1}^d \lambda_j} $$

This ratio is heavily used to choose $k$. For example, we might choose $k$ such that the cumulative explained variance reaches 95%.


## Part 21 & 22: Custom vs Library Implementations

Below we will execute our PCA experiments. We have modularized our experiment code into the `src/linear_algebra_and_stat/assignments/` directory.

- **Experiment 1** uses our internal custom `Matrix` class from Notebook 01. Since `matrix.py` is implemented for educational purposes and only explicitly solves the characteristic polynomial for 2x2 matrices, we limit our custom `Matrix` PCA to 2D datasets.
- **Experiment 2 & 3** utilize higher-dimensional datasets (4D Iris and 64D Digits). We use NumPy's `np.linalg.eigh` and `scikit-learn`'s `PCA` for these, demonstrating that the exact same theory holds and works equivalently.


In [ ]:
# Set up import paths
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from linear_algebra_and_stat.assignments import pca_experiment_2d
from linear_algebra_and_stat.assignments import pca_experiment_iris
from linear_algebra_and_stat.assignments import pca_experiment_image

%matplotlib inline


## Part 23: Experiment 1 - Synthetic 2D Data

We generate a 2D dataset with an elongated, rotated structure and correlated features.
We then use our custom `Matrix` class to center, build the covariance matrix, find the eigenvalues, and project the data.
Notice how the correlated, tilted cloud becomes perfectly aligned with the axes after PCA (decorrelation).


In [ ]:
# Extract results from the experiment module
X = pca_experiment_2d.generate_synthetic_2d_data(200)
mu, evals, evecs, Z, X_hat = pca_experiment_2d.custom_pca_2d(X)

print("Mean:", mu)
print("Eigenvalues:", evals)
print("Eigenvectors:\n", evecs)
print("\nCovariance before PCA (Sigma):\n", np.cov(X.T))
print("\nCovariance after PCA (approx diagonal Lambda):\n", np.cov(Z.T))

# Plotting the results directly in the notebook
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Original Data and Principal Directions
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.5, label='Original Data')
ax.scatter(mu[0], mu[1], color='red', marker='X', s=100, label='Mean')

for i in range(2):
    scale = np.sqrt(evals[i]) * 2
    vec = evecs[:, i] * scale
    ax.arrow(mu[0], mu[1], vec[0], vec[1], color='k', width=0.1, head_width=0.4, label=f'PC{i+1}' if i==0 else "")
    
ax.set_title("Original Data & Principal Directions")
ax.axis('equal')
ax.legend()

# Plot 2: Projected Data in PCA Space (Z)
ax = axes[1]
ax.scatter(Z[:, 0], Z[:, 1], alpha=0.5, color='orange')
ax.set_title("Data in PCA Space (Decorrelated)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.axis('equal')

# Plot 3: Reconstruction using only PC1
ax = axes[2]
ax.scatter(X[:, 0], X[:, 1], alpha=0.2, label='Original Data')
ax.scatter(X_hat[:, 0], X_hat[:, 1], alpha=0.8, color='green', label='Reconstruction (k=1)')

for i in range(len(X)):
    if i % 10 == 0:
        ax.plot([X[i, 0], X_hat[i, 0]], [X[i, 1], X_hat[i, 1]], 'r--', alpha=0.3)
        
ax.set_title("Reconstruction & Errors (PC1 Only)")
ax.axis('equal')
ax.legend()

plt.tight_layout()
plt.show()

## Part 24: Experiment 2 - Iris Dataset (NumPy vs Scikit-Learn)

We apply PCA to the 4D Iris dataset to project it down to 2D. 
We will compute this manually using numpy eigendecomposition and compare it with `sklearn.decomposition.PCA`.
*Note on Sign Ambiguity*: If $v$ is an eigenvector, $-v$ is also a valid eigenvector. `sklearn` and `numpy` might pick different signs, causing the projected scatter plots to be flipped along an axis. This is mathematically equivalent.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

# Run the computations from the module
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
target_names = iris.target_names

eigenvalues, eigenvectors, Z_custom, explained_variance_ratio_custom = pca_experiment_iris.custom_pca_iris(X_iris)

pca_sklearn = PCA(n_components=2)
Z_sklearn = pca_sklearn.fit_transform(X_iris)
X_hat_sklearn = pca_sklearn.inverse_transform(Z_sklearn)
W2_sklearn = pca_sklearn.components_.T # principal directions

print("--- Original Data Statistics ---")
print(f"Mean: {np.mean(X_iris, axis=0)}")
print(f"Covariance Matrix:\n{np.cov(X_iris.T)}")

print("\n--- Custom (NumPy) PCA ---")
print(f"Top 2 Eigenvalues: {eigenvalues[:2]}")
print(f"Explained Variance Ratio: {explained_variance_ratio_custom[:2]}")
print(f"Principal Directions (top 2):\n{eigenvectors[:, :2]}")

print("\n--- Scikit-Learn PCA ---")
print(f"Explained Variance (Eigenvalues): {pca_sklearn.explained_variance_}")
print(f"Explained Variance Ratio: {pca_sklearn.explained_variance_ratio_}")
print(f"Principal Directions (top 2):\n{W2_sklearn}")

print("\n--- Reconstruction Comparison ---")
# Reconstruct custom: X_hat = Z @ W2.T + mu
mu_iris = np.mean(X_iris, axis=0)
X_hat_custom = (Z_custom @ eigenvectors[:, :2].T) + mu_iris
print(f"Custom Reconstruction Error (MSE): {np.mean((X_iris - X_hat_custom)**2):.6f}")
print(f"Sklearn Reconstruction Error (MSE): {np.mean((X_iris - X_hat_sklearn)**2):.6f}")

# Plotting the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['navy', 'turquoise', 'darkorange']
lw = 2

# Plot 1: Custom PCA
ax = axes[0]
for color, i, target_name in zip(colors, [0, 1, 2], target_names):
    ax.scatter(Z_custom[y_iris == i, 0], Z_custom[y_iris == i, 1], color=color, alpha=.8, lw=lw, label=target_name)
ax.legend(loc='best', shadow=False, scatterpoints=1)
ax.set_title('PCA of IRIS dataset (Custom NumPy)')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')

# Plot 2: Scikit-learn PCA
ax = axes[1]
for color, i, target_name in zip(colors, [0, 1, 2], target_names):
    ax.scatter(Z_sklearn[y_iris == i, 0], Z_sklearn[y_iris == i, 1], color=color, alpha=.8, lw=lw, label=target_name)
ax.legend(loc='best', shadow=False, scatterpoints=1)
ax.set_title('PCA of IRIS dataset (Scikit-Learn)')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')

plt.tight_layout()
plt.show()

## Part 25: Experiment 3 - Image Dataset PCA

We use an image dataset (digits) where $n = 1797$ images and $d = 64$ pixels.
We apply PCA to the dataset and visualize the explained variance. Then, we reconstruct a single digit using an increasing number of principal components $k$ to visually demonstrate reconstruction loss and its relation to MSE.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data

cumulative_variance, pca_full = pca_experiment_image.compute_explained_variance(X_digits)

# Plot 1: Explained Variance
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance vs Components (Digits Dataset)')
plt.grid(True)
plt.show()

# Image Reconstruction with different k values
k_values = [2, 5, 10, 20, 50, 64]
img_idx = 0
original_image = X_digits[img_idx]

fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(16, 4))
axes[0].imshow(original_image.reshape(8, 8), cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i, k in enumerate(k_values):
    X_reconstructed, mse = pca_experiment_image.reconstruct_images(X_digits, k)
    recon_img = X_reconstructed[img_idx].reshape(8, 8)
    
    ax = axes[i + 1]
    ax.imshow(recon_img, cmap='gray')
    ax.set_title(f'k={k}\nMSE={mse:.2f}')
    ax.axis('off')
    
plt.tight_layout()
plt.show()

## Part 26: Optional Experiment - Single Image PCA (Low-Rank Approximation)

**Mathematical Distinction:**
In Part 25, we performed PCA across a *dataset of many images* (where one image = one sample point in $\mathbb{R}^{64}$). 
Here, we explore a different technique: performing PCA on a **single image matrix**. 
If we treat the $m$ rows of an image as $m$ distinct samples, and the $n$ columns as $n$ features, PCA will find correlations between the columns across the rows. This is mathematically equivalent to computing the **Truncated SVD** of the image matrix, forming a low-rank approximation. This is a common technique for image compression.

**Size Reduction & Reconstruction Error Theorem:**
When we compress a $m \times n$ image into $k$ principal components, we only need to store:
- $k$ principal component vectors (size $n$)
- $m$ projected coordinate vectors (size $k$)
- The mean vector (size $n$)
This requires storing $(k \times n) + (m \times k) + n$ values, which is often far smaller than $m \times n$.

Furthermore, the **Mean Squared Error (MSE)** of the reconstruction is mathematically proven to be exactly equal to the sum of the discarded variances. If we use the population covariance matrix ($\Sigma = \frac{1}{n}X_c^\intercal X_c$), then:
$$ \text{MSE} = \frac{1}{d} \sum_{j=k+1}^d \lambda_j $$
Let's visually prove this with a custom image loaded from our `data` directory.


In [ ]:
from linear_algebra_and_stat.assignments import pca_single_image

# Load custom test image from the data folder
X_img = pca_single_image.load_grayscale_image()
mu_img, evals_img, evecs_img, Z_img = pca_single_image.perform_single_image_pca(X_img)

k_list = [5,10, 50, 100]
results = pca_single_image.compress_and_verify(X_img, mu_img, evals_img, evecs_img, Z_img, k_list)

# Visualization
fig, axes = plt.subplots(1, len(k_list) + 1, figsize=(18, 5))
axes[0].imshow(X_img, cmap='gray')
axes[0].set_title(f"Original\nSize: {X_img.shape[0]}x{X_img.shape[1]}")
axes[0].axis('off')

for i, res in enumerate(results):
    ax = axes[i + 1]
    ax.imshow(res['X_hat'], cmap='gray')
    ax.set_title(f"k={res['k']}/{min(X_img.shape)} (-{res['size_reduction_pct']:.1f}% size)\nActual MSE: {res['actual_mse']:.4f}\nDiscarded Eval Sum: {res['discarded_eval_sum']:.4f}")
    ax.axis('off')

plt.tight_layout()
plt.show()

# Verification output
for res in results:
    diff = abs(res['actual_mse'] - res['discarded_eval_sum'])
    print(f"k={res['k']} -> Difference between MSE and Discarded Eigenvalues: {diff:.3e} (Effectively Zero!)")


## Summary

We have successfully derived PCA from first principles:
1. **Data as Geometry**: A dataset is a cloud of points.
2. **Mean & Centering**: We focus on the spread of the data around its center.
3. **Variance & Covariance**: We mathematically measure the spread and correlation between features.
4. **Covariance Matrix**: Encodes the entire geometric shape of the cloud.
5. **Optimization**: Maximizing projected variance ($w^\intercal \Sigma w$) leads directly to the Eigendecomposition problem ($\Sigma w = \lambda w$).
6. **Change of Basis**: Projecting onto the eigenvectors diagonalizes the covariance, completely decorrelating the features.
7. **Dimensionality Reduction**: Dropping eigenvectors with small eigenvalues optimally reduces dimensionality while minimizing reconstruction error.
